# `test_pipeline.py`

This notebook is organized into logical, executable sections from the corresponding Python source file.

## Setup

Import test dependencies and the modules under test.

In [ ]:
﻿"""Minimal smoke test for the pipeline."""

import os
import subprocess

import requests

import judge
from web_search import search_web




## Web-search fallback test

Verify searches fail safely when no API key is configured.

In [ ]:
def test_search_without_key_returns_empty_list():
    original = os.environ.get("WEB_SEARCH_API_KEY")
    os.environ.pop("WEB_SEARCH_API_KEY", None)
    try:
        assert search_web("Paris") == []
    finally:
        if original is not None:
            os.environ["WEB_SEARCH_API_KEY"] = original




## Python-version test

Verify the selected Python interpreter is supported.

In [ ]:
def test_preferred_python_is_supported():
    from data import preferred_python

    candidate = preferred_python()
    result = subprocess.run(
        [candidate, "-c", "import sys; assert (3, 11) <= sys.version_info[:2] < (3, 14); print(sys.version)"],
        capture_output=True,
        text=True,
        check=False,
    )
    assert result.returncode == 0, result.stderr or result.stdout




## Judge error-handling test

Verify API errors return an unverified result.

In [ ]:
def test_verify_answer_handles_api_errors(monkeypatch):
    monkeypatch.setenv("GROQ_API_KEY", "bad-key")

    def fake_post(*args, **kwargs):
        raise requests.exceptions.HTTPError("unauthorized")

    monkeypatch.setattr(judge.requests, "post", fake_post)

    result = judge.verify_answer(
        "Is Paris the capital of France?",
        "Yes, Paris is the capital of France.",
        [{"doc_id": "doc-1", "text": "Paris is the capital of France."}],
    )

    assert result["overall_verdict"] == "unverified"
    assert "Groq" in result["reason"]




## Direct smoke-test runner

Run the checks when this file is executed directly.

In [ ]:
if __name__ == "__main__":
    test_search_without_key_returns_empty_list()
    test_preferred_python_is_supported()
    test_verify_answer_handles_api_errors()
    print("minimal smoke test passed")
